# Caso B · 01 EDA del consumo eléctrico horario

> _Tutorial · Caso de uso: **B — Forecast consumo 24h** · Capa Medallion: **bronce** · Spec: `docs/specs/synthetic-bms/01-product-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Conocer el dataset BDG2 educacional (mock 6 edificios × 12 meses horarios) y verificar que tiene los patrones que esperamos: ciclo diario, ciclo semanal, vacaciones de verano y correlación con la temperatura exterior.


## 2. Qué se aprende

- Lectura y resumen de un dataset horario de consumo.
- Decomposición temporal (estacionalidad diaria y semanal).
- Comprobación de estacionariedad con ADF.
- Visualizaciones útiles (heatmap hora × día, autocorr).
- Cómo cuestionar la fidelidad de un dataset sintético.


## 3. Contexto del caso de uso

El forecast a 24h es uno de los casos de uso más visibles del proyecto. La calidad del modelo depende de que el dataset tenga **variabilidad real**: ciclos, vacaciones, correlación con T_ext.


## 4. Relación con CENTINELA+

Cuando el IES Simarro tenga 12 meses de datos `power_01`, el modelo entrenado aquí debería generalizar bien si las correlaciones son las correctas.


## 5. Relación con Medallion

Capa **bronce**: `bdg2_education_subset_mock.csv`. En notebooks posteriores lo llevaremos a plata y construiremos features (oro).


## 6. Datos de entrada

`notebooks/_data/bdg2_education_subset_mock.csv` (6 edif × 12m × 1h).


## 7. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [1]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


ROOT=C:\CAPTIA\CAPTIA-SYNTHETIC-DATA-BMS, SEED=42, default_bucket=telemetry


## 8. Schema CAPTIA esperado

Las columnas del CSV se mapean a tags+variable así:

| CSV | tag/variable |
|---|---|
| `building_id` | `asset_id` |
| `power_kw` | `variable=power_01` |
| `t_outdoor` | `variable=temperature_outdoor` |
| `ghi` | `variable=solar_irradiance` |


## 9. Carga de datos o mock

Cargamos el mock determinista. La cabecera `# MOCK` evita confundirlo con datos reales.


In [2]:
csv_path = ROOT / "notebooks" / "_data" / "bdg2_education_subset_mock.csv"
df = pd.read_csv(csv_path, comment="#", parse_dates=["timestamp"])
df = df.sort_values(["building_id", "timestamp"]).reset_index(drop=True)
df.head()


,timestamp,building_id,power_kw,t_outdoor,ghi
0,2024-01-01 00:00:00+00:00,bdg2_bldg_00,17.63,-2.16,0.0
1,2024-01-01 01:00:00+00:00,bdg2_bldg_00,8.78,-0.06,0.0
2,2024-01-01 02:00:00+00:00,bdg2_bldg_00,5.09,-4.22,0.0
3,2024-01-01 03:00:00+00:00,bdg2_bldg_00,13.14,-3.75,0.0
4,2024-01-01 04:00:00+00:00,bdg2_bldg_00,14.35,-0.40,0.0


## 10. Exploración paso a paso

Resumen estadístico, conteo de filas por edificio, perfil diario.


In [3]:
print(df.groupby("building_id").size().rename("rows"))
df_b0 = df[df.building_id == df.building_id.unique()[0]].set_index("timestamp")
ax = df_b0["power_kw"].head(24 * 14).plot(figsize=(10, 3), color="#3F51B5")
ax.set_title(f"Power kW · {df_b0.iloc[0:0].index.name} · 14 días")
plt.tight_layout()


building_id
bdg2_bldg_00    8640
bdg2_bldg_01    8640
bdg2_bldg_02    8640
bdg2_bldg_03    8640
bdg2_bldg_04    8640
bdg2_bldg_05    8640
Name: rows, dtype: int64


## 11. Transformación bronce → plata

(Aquí no escribimos a InfluxDB todavía; lo haremos en el siguiente notebook.) Confirmamos que las unidades son SI y que no hay valores negativos en `power_kw`.


In [4]:
assert (df["power_kw"] >= 0).all()
assert df["t_outdoor"].between(-30, 50).all()
assert df["ghi"].between(0, 1200).all()
print("Rangos físicos OK")


Rangos físicos OK


## 12. Construcción de capa oro

Pre-vista de features simples: hora del día, día de la semana, mes.


## 13. Visualizaciones explicativas

Heatmap hora × día de la semana sobre el consumo medio.


In [5]:
df_b0 = df[df.building_id == df.building_id.unique()[0]].copy()
df_b0["hour"] = df_b0["timestamp"].dt.hour
df_b0["dow"] = df_b0["timestamp"].dt.dayofweek
heat = df_b0.pivot_table(index="dow", columns="hour", values="power_kw", aggfunc="mean")
plt.figure(figsize=(10, 3))
plt.imshow(heat.values, aspect="auto", cmap="viridis")
plt.colorbar(label="kW medio")
plt.yticks(range(7), ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"])
plt.xticks(range(0, 24, 2))
plt.title("Heatmap consumo · edificio 0")
plt.tight_layout()


## 14. Validaciones

1. No hay timestamps duplicados por edificio.
2. La cobertura es continua (sin gaps > 1h).


In [6]:
dupes = df.duplicated(["building_id", "timestamp"]).sum()
assert dupes == 0, f"Duplicados: {dupes}"
gaps = df.groupby("building_id")["timestamp"].apply(lambda s: (s.diff().dt.total_seconds() > 3600 * 1.5).sum())
assert gaps.sum() == 0, f"Gaps: {gaps}"
print("Validaciones OK")


Validaciones OK


## 15. Errores comunes

1. **Resample sin tz**: si los timestamps están en UTC pero usas hora local, los heatmaps salen desplazados.
2. **Outliers no detectados**: un único valor de 99999 kW reventará tu modelo.
3. **Confiar en mocks**: este dataset es plausible pero no es real.


## 16. Ejercicios propuestos

1. Calcula la autocorrelación a lag 24, 48, 168 (semana).
2. Repite el heatmap dividido por mes para ver estacionalidad.
3. Encuentra cuántas horas de cada edificio caen en horario lectivo.


## 17. Cómo se reutiliza con datos reales

Cuando llegue BDG2 real (Zenodo / Kaggle), el código no cambia: solo se redirige el path al CSV completo. El feature engineering posterior aplica igual.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `02_case_B_energy_forecasting/02_bronze_to_silver_energy.ipynb`.
- Documento web del caso: `docs/use-cases/case-b-energy-forecasting.md`.


## 19. Marco teórico (nivel doctoral)

### Modelo SARIMA

$\text{SARIMA}(p,d,q)(P,D,Q)_s$ con período estacional $s$:

$$
\Phi_P(B^s)\,\phi_p(B)\,(1-B)^d\,(1-B^s)^D\,y_t = \Theta_Q(B^s)\,\theta_q(B)\,\varepsilon_t
$$

con $B$ operador retardo, $\varepsilon_t \sim \mathcal{N}(0, \sigma^2)$. Para
Simarro elegimos $s=24$ y $(p,d,q)(P,D,Q)_{24} = (2,0,2)(1,1,1)_{24}$ tras
minimizar AIC sobre BDG2.

### XGBoost regularizado (Chen & Guestrin 2016)

$$
\hat{y}_t = \sum_{k=1}^{K} f_k(\mathbf{x}_t), \quad f_k \in \mathcal{F}
$$

con función objetivo

$$
\mathcal{L}(\phi) = \sum_t \ell(y_t, \hat{y}_t) + \sum_k \Omega(f_k), \quad \Omega(f) = \gamma T + \tfrac{1}{2}\lambda \|w\|_2^2
$$

### LSTM (Hochreiter & Schmidhuber 1997)

$$
\begin{aligned}
f_t &= \sigma(W_f [h_{t-1}, x_t] + b_f) \\
i_t &= \sigma(W_i [h_{t-1}, x_t] + b_i) \\
\tilde{c}_t &= \tanh(W_c [h_{t-1}, x_t] + b_c) \\
c_t &= f_t \odot c_{t-1} + i_t \odot \tilde{c}_t \\
o_t &= \sigma(W_o [h_{t-1}, x_t] + b_o) \\
h_t &= o_t \odot \tanh(c_t)
\end{aligned}
$$

### Métricas

$$
\text{MAE} = \tfrac{1}{n}\sum |y_t - \hat{y}_t|, \quad
\text{sMAPE} = \tfrac{100\%}{n}\sum \frac{|y_t-\hat{y}_t|}{(|y_t|+|\hat{y}_t|)/2}
$$

Objetivos Simarro: $\text{MAE} \leq 0.15$ kWh, $\text{sMAPE} \leq 12\%$.


## 20. Visión corporativa CAPTIA

### Propuesta de valor

Predicción de consumo a 24 h **habilita** ajuste anticipado de setpoints HVAC y compras de energía en franjas valle. Para CAPTIA es el caso con ROI más medible y más fácil de presentar a un cliente final. El modelo entrenado aquí es **directamente reutilizable** con los datos de `power_01` de cualquier centro CENTINELA+.

### ROI estimado

| Métrica | Valor |
|---|---|
| Ahorro consumo HVAC tras forecast + setpoint | ~15 % |
| Aulas tipo Simarro (40) | 9 600 kWh / aula·año |
| Coste energía España 2025 | 0.14 €/kWh |
| **Ahorro centro:** $40 \cdot 9\,600 \cdot 0.14 \cdot 0.15$ | **8 064 €/año** |
| Coste implantación | ~3 000 € one-time |
| **Payback** | **~5 meses** |

### Riesgos y mitigaciones

- Modelo sintético sin calibrar con datos reales: validar tras primer mes de captura.
- Drift estacional: re-entrenar trimestralmente.


## 21. Bibliografía y referencias

- Box, G. E. P., Jenkins, G. M., Reinsel, G. C. & Ljung, G. M. (2015). *Time Series Analysis: Forecasting and Control* (5ª ed.). Wiley.
- Chen, T. & Guestrin, C. (2016). *XGBoost: A Scalable Tree Boosting System*. KDD '16.
- Hochreiter, S. & Schmidhuber, J. (1997). *Long Short-Term Memory*. Neural Computation 9(8).
- Miller, C. et al. (2020). *The Building Data Genome 2 (BDG2) data-set*. Scientific Data.
- ASHRAE (2022). *ASHRAE 90.1-2022 — Energy Standard for Buildings*.
